# ARIMA(1,0,1)

Locked for every platinum2 notebook:

| Rule | Value |
|------|--------|
| Task | Forecast **delayed MW** (EIA planned stock; GIA is an overlay). |
| Headline | 12-month-ahead RMSE vs seasonal naive |
| Sanity | 3-month-ahead RMSE |
| Val | origins whose **target year ≤ 2023** (12m) / 2023 origins for 3m |
| Test | **2024 sealed** |
| Split shuffle | **No** |

This is not Platinum 1 project-row months of slip.

```text
CPU:  .\.venv\Scripts\Activate.ps1
GPU (TimesFM):  wsl … python scripts/run_platinum2_all.py --only timesfm
```

Trainer: `fit_arima`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO = Path.cwd().resolve()
for p in [REPO, *REPO.parents]:
    if (p / "src" / "modeling").exists():
        REPO = p
        break
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

MODEL_ID = "arima"
RESULTS = REPO / "platinum2" / "results" / MODEL_ID
(RESULTS / "models").mkdir(parents=True, exist_ok=True)
(RESULTS / "plots").mkdir(parents=True, exist_ok=True)

In [ ]:
from src.modeling.platinum2.trainers import fit_arima

m = fit_arima()
keys = [k for k in m if k not in ("history", "folds", "_model")]
print(json.dumps({k: m[k] for k in keys}, indent=2, default=str))

In [ ]:
keys = ("model", "status", "rmse", "mape", "rmse_h3", "mape_h3", "n_folds_h12", "n_folds_h3", "beats_seasonal_naive", "reason")
pub = {{k: m.get(k) for k in keys}}
(RESULTS / "metrics.json").write_text(json.dumps(pub, indent=2, default=str), encoding="utf-8")
pd.DataFrame(m.get("history") or []).to_csv(RESULTS / "history.csv", index=False)
print("wrote", RESULTS / "metrics.json")